<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/mid_module_exam_01_03_sem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q: 32
Problem Statement
A small online electronics retailer wants to flag negative product reviews for the customer-support team. You must build a binary sentiment classifier (positive vs negative) that uses a TF-IDF representation and compares Multinomial Naive Bayes against a Linear SVM, while making sure the preprocessing does not destroy negation cues.

Dataset
A tiny labelled snippet (label 1 = positive, 0 = negative) is provided inline; treat it as the seed of a much larger dataset and train on it as-is.

text,label
The charger works perfectly and arrived early,1
Product is good and battery lasts a full day,1
Fast shipping; the headphones sound great,1
The cable is not durable and stopped working in two days,0
Never buying from this seller again terrible service,0
The earbuds were damaged on arrival and support is slow,0
Decent build quality and easy to use,1
The item is not what was advertised and the box was broken,0
Tasks
Implement a Python pipeline that:

Loads the inline data into a DataFrame.
Cleans text (lowercasing, punctuation/number removal) but keeps negation words such as not, never, and neither.
Uses an 80/20 stratified train/test split.
Builds TF-IDF features with TfidfVectorizer (fit_transform on train, transform on test).
Trains both a MultinomialNB and a LinearSVC, and prints accuracy plus a confusion matrix for each on the test set.
Briefly justify why the negation-aware stop-word handling matters for sentiment performance, citing one example phrase from the dataset.

Expected Output
For each model, the test-set accuracy and a 2x2 confusion matrix printed to stdout, plus a 2-3 sentence written justification of the negation-aware preprocessing.

In [ ]:
import pandas as pd
import re
from io import StringIO

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix

# -----------------------------
# Load inline dataset
# -----------------------------
data = """text,label
The charger works perfectly and arrived early,1
Product is good and battery lasts a full day,1
Fast shipping; the headphones sound great,1
The cable is not durable and stopped working in two days,0
Never buying from this seller again terrible service,0
The earbuds were damaged on arrival and support is slow,0
Decent build quality and easy to use,1
The item is not what was advertised and the box was broken,0
"""

df = pd.read_csv(StringIO(data))

# -----------------------------
# Text cleaning
# Keep negation words like:
# not, never, neither, no
# -----------------------------
NEGATION_WORDS = {"not", "never", "neither", "no"}

# Remove negation words from stopword list
custom_stopwords = list(set(ENGLISH_STOP_WORDS) - NEGATION_WORDS)

def clean_text(text):
    text = text.lower()

    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["clean_text"] = df["text"].apply(clean_text)

# -----------------------------
# Train/test split (80/20)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# -----------------------------
# TF-IDF Vectorization
# -----------------------------
vectorizer = TfidfVectorizer(stop_words=custom_stopwords)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# -----------------------------
# Model 1: Multinomial Naive Bayes
# -----------------------------
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

nb_preds = nb_model.predict(X_test_tfidf)

print("=== Multinomial Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, nb_preds))
print("Confusion Matrix:")
print(confusion_matrix(y_test, nb_preds))

# -----------------------------
# Model 2: Linear SVM
# -----------------------------
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

svm_preds = svm_model.predict(X_test_tfidf)

print("\n=== Linear SVM ===")
print("Accuracy:", accuracy_score(y_test, svm_preds))
print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_preds))

# -----------------------------
# Justification
# -----------------------------
print("\nWhy negation-aware preprocessing matters:")
print(
    "Negation words such as 'not' and 'never' carry critical sentiment information. "
    "If they are removed as stop words, phrases like "
    "'not durable' or 'Never buying from this seller again' may incorrectly appear "
    "positive or neutral because only words like 'durable' or 'buying' remain. "
    "Keeping negation terms helps the classifier preserve the true negative sentiment."
)

=== Multinomial Naive Bayes ===
Accuracy: 0.5
Confusion Matrix:
[[1 0]
 [1 0]]

=== Linear SVM ===
Accuracy: 0.5
Confusion Matrix:
[[0 1]
 [0 1]]

Why negation-aware preprocessing matters:
Negation words such as 'not' and 'never' carry critical sentiment information. If they are removed as stop words, phrases like 'not durable' or 'Never buying from this seller again' may incorrectly appear positive or neutral because only words like 'durable' or 'buying' remain. Keeping negation terms helps the classifier preserve the true negative sentiment.


In [ ]:
#solution in exam
import io
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix
import re

data = '''text,label
The charger works perfectly and arrived early,1
Product is good and battery lasts a full day,1
Fast shipping; the headphones sound great,1
The cable is not durable and stopped working in two days,0
Never buying from this seller again terrible service,0
The earbuds were damaged on arrival and support is slow,0
Decent build quality and easy to use,1
The item is not what was advertised and the box was broken,0'''

df = pd.read_csv(io.StringIO(data))

# Lowercase and strip punctuation/numbers while keeping negation words intact
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['clean'] = df['text'].apply(clean)

# Custom stop list: standard English minus negation cues
NEGATIONS = {'not', 'never', 'neither', 'no'}
custom_stop = list(ENGLISH_STOP_WORDS - NEGATIONS)

X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

vec = TfidfVectorizer(stop_words=custom_stop)
X_train_vec = vec.fit_transform(X_train)
X_test_vec  = vec.transform(X_test)

for name, model in [('MultinomialNB', MultinomialNB()), ('LinearSVC', LinearSVC())]:
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    print(f'{name} accuracy:', accuracy_score(y_test, preds))
    print(f'{name} confusion matrix:\n', confusion_matrix(y_test, preds))


MultinomialNB accuracy: 0.5
MultinomialNB confusion matrix:
 [[1 0]
 [1 0]]
LinearSVC accuracy: 0.5
LinearSVC confusion matrix:
 [[0 1]
 [0 1]]


Problem Statement
A medical-imaging startup wants to clean low-quality scanned digit images before downstream classification. Build a small denoising autoencoder in PyTorch on the MNIST dataset that takes noisy 28x28 images as input and reconstructs clean originals.

Dataset
Use torchvision.datasets.MNIST (train split) for both inputs and targets. Generate noisy inputs on-the-fly by adding Gaussian noise of standard deviation 0.3 to each clean image and clipping back to [0, 1]. Treat the original clean image as the reconstruction target.

Tasks
Define a denoising autoencoder with:

An encoder of fully connected layers reducing 784 to a latent vector of size 64 with ReLU activations.
A decoder of fully connected layers expanding 64 back to 784 with ReLU activations and a final sigmoid output.
Train the model with mean squared error loss between the reconstruction and the clean image, using the Adam optimiser, for at least 5 epochs.

After training, demonstrate visually or numerically that the model removes noise using a real held-out validation batch created from the MNIST training split (e.g., print the average MSE on the held-out batch for noisy vs clean images, before and after passing through the model).

Expected Output
A complete PyTorch implementation of the denoising autoencoder.
A short numeric or printed comparison showing that reconstructed outputs are closer to the clean targets than the noisy inputs were.
Submission Format
Paste your complete PyTorch code inline, followed by 2-3 sentences explaining how the latent vector lets the network filter noise.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# -----------------------------
# Configuration
# -----------------------------
BATCH_SIZE = 128
LATENT_DIM = 64
EPOCHS = 5
LEARNING_RATE = 1e-3
NOISE_STD = 0.3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

# -----------------------------
# Load MNIST Dataset
# -----------------------------
transform = transforms.ToTensor()

mnist_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Split into train and validation
train_size = int(0.9 * len(mnist_dataset))
val_size = len(mnist_dataset) - train_size

train_dataset, val_dataset = random_split(
    mnist_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# -----------------------------
# Function to Add Gaussian Noise
# -----------------------------
def add_noise(images, noise_std=0.3):
    noise = torch.randn_like(images) * noise_std
    noisy_images = images + noise
    noisy_images = torch.clamp(noisy_images, 0.0, 1.0)
    return noisy_images

# -----------------------------
# Denoising Autoencoder
# -----------------------------
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super(DenoisingAutoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, LATENT_DIM),
            nn.ReLU()
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(LATENT_DIM, 128),
            nn.ReLU(),

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)

        latent = self.encoder(x)

        reconstructed = self.decoder(latent)

        reconstructed = reconstructed.view(-1, 1, 28, 28)

        return reconstructed

# -----------------------------
# Initialize Model
# -----------------------------
model = DenoisingAutoencoder().to(DEVICE)

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    for images, _ in train_loader:

        images = images.to(DEVICE)

        # Create noisy input
        noisy_images = add_noise(images, NOISE_STD)

        # Forward pass
        outputs = model(noisy_images)

        # Reconstruction loss against CLEAN targets
        loss = criterion(outputs, images)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.6f}")

# -----------------------------
# Evaluation on Held-Out Batch
# -----------------------------
model.eval()

with torch.no_grad():

    val_images, _ = next(iter(val_loader))

    val_images = val_images.to(DEVICE)

    # Create noisy validation images
    noisy_val_images = add_noise(val_images, NOISE_STD)

    # Reconstructed images
    reconstructed_images = model(noisy_val_images)

    # MSE BEFORE denoising
    noisy_mse = criterion(noisy_val_images, val_images).item()

    # MSE AFTER denoising
    reconstructed_mse = criterion(reconstructed_images, val_images).item()

    print("\n--- Denoising Performance ---")
    print(f"MSE between noisy and clean images: {noisy_mse:.6f}")
    print(f"MSE between reconstructed and clean images: {reconstructed_mse:.6f}")

    if reconstructed_mse < noisy_mse:
        print("The autoencoder successfully reduced noise.")
    else:
        print("The autoencoder did not improve reconstruction.")

'The latent vector acts as a compressed representation of the digit, forcing the network to keep only the most important structural features while discarding random pixel-level noise. Because the encoder reduces the image to a compact 64-dimensional space, the model learns general digit patterns instead of memorising noisy details. The decoder then reconstructs a cleaner version of the image from this compact representation.'

Using device: cpu
Epoch [1/5] Loss: 0.064147
Epoch [2/5] Loss: 0.036173
Epoch [3/5] Loss: 0.028226
Epoch [4/5] Loss: 0.024642
Epoch [5/5] Loss: 0.022514

--- Denoising Performance ---
MSE between noisy and clean images: 0.046400
MSE between reconstructed and clean images: 0.022191
The autoencoder successfully reduced noise.


'The latent vector acts as a compressed representation of the digit, forcing the network to keep only the most important structural features while discarding random pixel-level noise. Because the encoder reduces the image to a compact 64-dimensional space, the model learns general digit patterns instead of memorising noisy details. The decoder then reconstructs a cleaner version of the image from this compact representation.'

In [ ]:
#SOlution in exam
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'

transform = transforms.ToTensor()  # gives [0,1] float images
full_train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Create a real held-out validation split from the MNIST training split
train_size = 55000
val_size = len(full_train_ds) - train_size
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_train_ds, [train_size, val_size], generator=generator)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128, shuffle=False)

class DenoisingAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64),  nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 256),  nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = DenoisingAE().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(5):
    model.train()
    total = 0.0

    for x, _ in train_dl:
        x = x.view(x.size(0), -1).to(device)
        noisy = torch.clamp(x + 0.3 * torch.randn_like(x), 0.0, 1.0)

        recon = model(noisy)
        loss = loss_fn(recon, x)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item() * x.size(0)

    print(f'epoch {epoch+1} mse {total/len(train_ds):.4f}')

# Held-out validation comparison
model.eval()
x, _ = next(iter(val_dl))
x = x.view(x.size(0), -1).to(device)
noisy = torch.clamp(x + 0.3 * torch.randn_like(x), 0.0, 1.0)

with torch.no_grad():
    recon = model(noisy)

print('Validation MSE noisy vs clean :', loss_fn(noisy, x).item())
print('Validation MSE recon  vs clean :', loss_fn(recon, x).item())


100%|██████████| 9.91M/9.91M [00:00<00:00, 38.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.04MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.66MB/s]


epoch 1 mse 0.0500
epoch 2 mse 0.0243
epoch 3 mse 0.0197
epoch 4 mse 0.0173
epoch 5 mse 0.0157
Validation MSE noisy vs clean : 0.04630979895591736
Validation MSE recon  vs clean : 0.01515779085457325


In [ ]:
# Using device: cpu
# Epoch [1/5] Loss: 0.064147
# Epoch [2/5] Loss: 0.036173
# Epoch [3/5] Loss: 0.028226
# Epoch [4/5] Loss: 0.024642
# Epoch [5/5] Loss: 0.022514

# --- Denoising Performance ---
# MSE between noisy and clean images: 0.046400
# MSE between reconstructed and clean images: 0.022191
# The autoencoder successfully reduced noise.